# #️⃣ Day 02a: Hash Indexing — Hash Tables, Bitmap Indexes, Comparison

---

## 🎯 What You'll Master Today
- Hash index internals (static & extendible)
- Bitmap indexes for low-cardinality columns
- B+ Tree vs Hash vs Bitmap comparison
- Index selection guidelines

---

```
╔══════════════════════════════════════════════════════════════════╗
║                   HASH INDEX STRUCTURE                           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  key → hash(key) → bucket_number → bucket → row pointer        ║
║                                                                  ║
║  Bucket 0: [ (Alice, ptr1) ]                                    ║
║  Bucket 1: [ (Bob, ptr2), (Eve, ptr5) ]  ← chaining            ║
║  Bucket 2: [ (Charlie, ptr3) ]                                  ║
║  Bucket 3: [ (Diana, ptr4) ]                                    ║
║                                                                  ║
║  ✓ O(1) exact match — fastest for equality!                    ║
║  ✗ No range queries (hash destroys order)                       ║
║  ✗ No ORDER BY, no partial match, no LIKE                       ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Hash Index Simulation
# ============================================

class HashIndex:
    def __init__(self, num_buckets=8):
        self.num_buckets = num_buckets
        self.buckets = [[] for _ in range(num_buckets)]
        self.io_count = 0
    
    def _hash(self, key):
        return hash(key) % self.num_buckets
    
    def insert(self, key, row_ptr):
        bucket_id = self._hash(key)
        self.buckets[bucket_id].append((key, row_ptr))
    
    def search(self, key):
        self.io_count = 0
        bucket_id = self._hash(key)
        self.io_count = 1  # 1 I/O to read bucket page
        
        for k, ptr in self.buckets[bucket_id]:
            if k == key:
                return ptr, self.io_count
        return None, self.io_count
    
    def display(self):
        for i, bucket in enumerate(self.buckets):
            entries = ", ".join(f"{k}→{p}" for k, p in bucket) if bucket else "(empty)"
            print(f"    Bucket {i}: [{entries}]")

hi = HashIndex(num_buckets=4)
data = [("Alice", "row1"), ("Bob", "row2"), ("Charlie", "row3"),
        ("Diana", "row4"), ("Eve", "row5"), ("Frank", "row6")]

for key, ptr in data:
    hi.insert(key, ptr)

print("  Hash Index (4 buckets):")
hi.display()

print("\n  Lookups:")
for name in ["Charlie", "Eve", "Zara"]:
    result, ios = hi.search(name)
    status = f"→ {result}" if result else "NOT FOUND"
    print(f"    search('{name}'): {status} ({ios} I/O)")

print("\n  💡 Always O(1) I/O for equality — but can't do WHERE name > 'C'!")

In [ ]:
# ============================================
# 2. Extendible Hashing (grows dynamically)
# ============================================

print("""
  ╔════════════════════════════════════════════════════════════════╗
  ║            EXTENDIBLE HASHING                                  ║
  ╠════════════════════════════════════════════════════════════════╣
  ║                                                                ║
  ║  Problem: Static hash → overflow chains → degraded O(1)      ║
  ║  Solution: Extendible hashing → split buckets as needed       ║
  ║                                                                ║
  ║  Global Depth = 2 (use last 2 bits)                           ║
  ║      Directory:                                                ║
  ║      00 → Bucket A [items with ...00]                         ║
  ║      01 → Bucket B [items with ...01]                         ║
  ║      10 → Bucket C [items with ...10]                         ║
  ║      11 → Bucket D [items with ...11]                         ║
  ║                                                                ║
  ║  When bucket overflows:                                        ║
  ║  • Split bucket, increase local depth                         ║
  ║  • May double directory (if local depth > global depth)       ║
  ║  • Only redistributes ONE bucket's entries                     ║
  ║                                                                ║
  ╚════════════════════════════════════════════════════════════════╝
""")

class ExtendibleHash:
    def __init__(self, bucket_size=2):
        self.global_depth = 1
        self.bucket_size = bucket_size
        self.directory = {0: [], 1: []}
    
    def _hash_bits(self, key, depth):
        return hash(key) & ((1 << depth) - 1)
    
    def insert(self, key):
        idx = self._hash_bits(key, self.global_depth)
        bucket = self.directory[idx]
        if len(bucket) < self.bucket_size:
            bucket.append(key)
        else:
            print(f"    Bucket {idx} full! Need split (not implemented in demo)")
            bucket.append(key)  # simplified
    
    def display(self):
        print(f"    Global Depth: {self.global_depth}")
        for idx in sorted(self.directory.keys()):
            bits = format(idx, f'0{self.global_depth}b')
            print(f"    {bits} → {self.directory[idx]}")

eh = ExtendibleHash(bucket_size=2)
for k in ["Alice", "Bob", "Charlie"]:
    eh.insert(k)
print("  Extendible Hash:")
eh.display()

In [ ]:
# ============================================
# 3. Bitmap Index — for low-cardinality columns
# ============================================

print("  Bitmap Index: perfect for columns with few distinct values\n")

employees = [
    {"id": 1, "gender": "M", "dept": "Eng",  "active": True},
    {"id": 2, "gender": "F", "dept": "Eng",  "active": True},
    {"id": 3, "gender": "M", "dept": "Sales","active": False},
    {"id": 4, "gender": "F", "dept": "Sales","active": True},
    {"id": 5, "gender": "M", "dept": "Eng",  "active": True},
    {"id": 6, "gender": "F", "dept": "HR",   "active": False},
]

# Build bitmap indexes
def build_bitmap(data, column):
    values = sorted(set(row[column] for row in data))
    bitmaps = {}
    for val in values:
        bitmaps[val] = [1 if row[column] == val else 0 for row in data]
    return bitmaps

gender_bm = build_bitmap(employees, "gender")
dept_bm = build_bitmap(employees, "dept")
active_bm = build_bitmap(employees, "active")

print("  Gender Bitmap:")
for val, bits in gender_bm.items():
    print(f"    {val}: {bits}")

print("\n  Department Bitmap:")
for val, bits in dept_bm.items():
    print(f"    {val:5}: {bits}")

# Bitmap AND operation: WHERE gender='F' AND dept='Eng'
print("\n  Query: WHERE gender='F' AND dept='Eng'")
f_bits = gender_bm["F"]
eng_bits = dept_bm["Eng"]
result_bits = [a & b for a, b in zip(f_bits, eng_bits)]
print(f"    F:       {f_bits}")
print(f"    Eng:     {eng_bits}")
print(f"    F & Eng: {result_bits}")
matches = [employees[i] for i, b in enumerate(result_bits) if b]
print(f"    Result: {[m['id'] for m in matches]} → {[m['gender']+'/'+m['dept'] for m in matches]}")
print("\n  💡 Bitmap: AND/OR/NOT are just bitwise ops → extremely fast for OLAP!")

In [ ]:
# ============================================
# 4. Index Type Comparison
# ============================================

print("""
  ┌────────────────────┬──────────────┬──────────────┬──────────────┐
  │ Feature             │ B+ Tree      │ Hash         │ Bitmap       │
  ├────────────────────┼──────────────┼──────────────┼──────────────┤
  │ Equality (=)        │ O(log n)     │ O(1) ✓ best │ O(1) bitwise │
  │ Range (>, <, BTWN)  │ O(log n + k) │ ✗ N/A       │ ✗ N/A        │
  │ ORDER BY            │ ✓ Yes        │ ✗ No         │ ✗ No         │
  │ LIKE 'abc%'         │ ✓ Prefix     │ ✗ No         │ ✗ No         │
  │ Low cardinality     │ Wasteful     │ Wasteful     │ ✓ Perfect    │
  │ High cardinality    │ ✓ Good       │ ✓ Best       │ ✗ Too many   │
  │ Concurrent writes   │ ✓ Good       │ ✗ Lock heavy │ ✗ Poor       │
  │ Space               │ Medium       │ Medium       │ Tiny (bits!) │
  ├────────────────────┼──────────────┼──────────────┼──────────────┤
  │ Best for            │ General      │ Key lookups  │ Warehousing  │
  │                     │ purpose      │ hash joins   │ OLAP queries │
  └────────────────────┴──────────────┴──────────────┴──────────────┘

  Quick Decision Guide:
  ┌─────────────────────────────────────────────────────────┐
  │ Need range queries?          → B+ Tree                 │
  │ Only equality + high card.?  → Hash Index              │
  │ Low cardinality + OLAP?      → Bitmap                  │
  │ Not sure?                    → B+ Tree (safe default)  │
  └─────────────────────────────────────────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Hash vs B+ Tree index? | Hash: O(1) equality only. B+ Tree: O(log n) but supports range/order |
| 2 | What is a bitmap index? | Bit vector per distinct value. AND/OR = bitwise ops. Great for OLAP |
| 3 | When to use hash index? | Exact-match lookups on high-cardinality columns (e.g., user_id) |
| 4 | Why not hash for everything? | No range queries, no ordering, no partial match, no LIKE |
| 5 | Extendible vs linear hashing? | Extendible: directory doubles. Linear: split pointer advances linearly |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Hash: O(1) equality, no range support                │
## │  • Bitmap: bit vectors + bitwise AND/OR → fast OLAP     │
## │  • B+ Tree: universal default (range + order + equality)│
## │  • Extendible hashing: grows without full rehash        │
## │  • Choose index type based on query patterns!           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Query Optimization** — EXPLAIN, query plans, optimization tips